# GPT2 Masked Multi-head Self-attention详解

## 环境准备

本案例的运行环境为：

| Python | MindSpore |
| :----- | :-------- |
| 3.10   | 2.7.0     |

如果你在如[昇思大模型平台](https://xihe.mindspore.cn/training-projects)、[华为云ModelArts](https://www.huaweicloud.com/product/modelarts.html)、[启智社区](https://openi.pcl.ac.cn/)等算力平台的Jupyter在线编程环境中运行本案例，可取消如下代码的注释，进行依赖库安装：

In [1]:
import numpy as np
import mindspore
from mindspore import nn, ops, Tensor, mint

/usr/local/python3.10.14/lib/python3.10/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/python3.10.14/lib/python3.10/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/python3.10.14/lib/python3.10/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/python3.10.14/lib/python3.10/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)


## GPT-2 Self-attention: 1- Creating queries, keys, and values

![gpt2-self-attention-3.png](https://jalammar.github.io/images/gpt2/gpt2-self-attention-3.png)

In [2]:
batch_size = 1
seq_len = 10
embed_dim = 768

# input x: (1, 10, 768)
x = Tensor(np.random.randn(batch_size, seq_len, embed_dim), mindspore.float32)

In [3]:
from mindspore.nn import Dense

# 替换 Conv1D 为标准的 Dense 层（在 Transformer 中更常用）
c_attn = Dense(embed_dim, 3 * embed_dim, has_bias=True)

# 使用 MindSpore 的 split 操作
output = c_attn(x)  # shape: (1, 10, 768 * 3)
query, key, value = ops.split(output, split_size_or_sections=embed_dim, axis=2)

print(f"Query shape: {query.shape}")   # (1, 10, 768)
print(f"Key shape: {key.shape}")       # (1, 10, 768)  
print(f"Value shape: {value.shape}")   # (1, 10, 768)

[WARNING] CORE(32198,ffff93c8b640,python):2025-11-24-11:00:52.065.913 [mindspore/core/utils/ms_context.cc:535] GetJitLevel] Set jit level to O2 for rank table startup method.


Query shape: (1, 10, 768)
Key shape: (1, 10, 768)
Value shape: (1, 10, 768)


![gpt2-self-attention-split-attention-heads-1.png](https://jalammar.github.io/images/gpt2/gpt2-self-attention-split-attention-heads-1.png)

![gpt2-self-attention-split-attention-heads-2.png](https://jalammar.github.io/images/gpt2/gpt2-self-attention-split-attention-heads-2.png)

In [4]:
def split_heads(tensor, num_heads, attn_head_size):
    """
    Splits hidden_size dim into attn_head_size and num_heads
    """
    # (batch_size, seq_len, hidden_size) --> (batch_size, seq_len, num_heads, attn_head_size)
    new_shape = tensor.shape[:-1] + (num_heads, attn_head_size)
    tensor = tensor.view(new_shape)
    # (batch_size, seq_len, num_heads, attn_head_size) --> (batch_size, num_heads, seq_len, attn_head_size)
    return ops.transpose(tensor, (0, 2, 1, 3))  

In [5]:
num_heads = 12
head_dim = embed_dim // num_heads

# (1, 10, 768) --> (1, 10, 12, 64) --> (1, 12, 10, 64)
query = split_heads(query, num_heads, head_dim)
key = split_heads(key, num_heads, head_dim)
value = split_heads(value, num_heads, head_dim)

query.shape, key.shape, value.shape

((1, 12, 10, 64), (1, 12, 10, 64), (1, 12, 10, 64))

## GPT-2 Self-attention: 2- Scoring

![gpt2-self-attention-scoring.png](https://jalammar.github.io/images/gpt2/gpt2-self-attention-scoring.png)

![](https://jalammar.github.io/images/gpt2/gpt2-self-attention-scoring-2.png)

In [6]:
# qk点积
# q: (1, 12, 10, 64), k^T: (1, 12, 64, 10)
# attn_weights: (1, 12, 10, 10)
attn_weights = ops.matmul(query, key.swapaxes(-1, -2))

attn_weights.shape

(1, 12, 10, 10)

![](https://jalammar.github.io/images/gpt2/transformer-decoder-attention-mask-dataset.png)

In [7]:
# diagonal matrix to implement masked multi-head attention
# To ensure not to attend to future information
max_positions = seq_len

bias = Tensor(np.tril(np.ones((max_positions, max_positions))).reshape(
              (1, 1, max_positions, max_positions)), mindspore.bool_)
bias

Tensor(shape=[1, 1, 10, 10], dtype=Bool, value=
[[[[ True, False, False ... False, False, False],
   [ True,  True, False ... False, False, False],
   [ True,  True,  True ... False, False, False],
   ...
   [ True,  True,  True ...  True, False, False],
   [ True,  True,  True ...  True,  True, False],
   [ True,  True,  True ...  True,  True,  True]]]])

![](https://jalammar.github.io/images/gpt2/queries-keys-attention-mask.png)

![](https://jalammar.github.io/images/gpt2/transformer-attention-mask.png)

In [8]:
attn_weights = attn_weights / ops.sqrt(ops.scalar_to_tensor(value.shape[-1]))
query_length, key_length = query.shape[-2], key.shape[-2]
causal_mask = bias[:, :, key_length - query_length: key_length, :key_length].bool()
mask_value = Tensor(np.finfo(np.float32).min, dtype=attn_weights.dtype)
attn_weights = ops.where(causal_mask, attn_weights, mask_value)

In [9]:
np.finfo(np.float32).min

-3.4028235e+38

In [10]:
attn_weights[0, 0]

Tensor(shape=[10, 10], dtype=Float32, value=
[[ 4.36844140e-01, -3.40282347e+38, -3.40282347e+38 ... -3.40282347e+38, -3.40282347e+38, -3.40282347e+38],
 [ 3.22000295e-01,  5.70516859e-04, -3.40282347e+38 ... -3.40282347e+38, -3.40282347e+38, -3.40282347e+38],
 [-2.47207835e-01, -2.36715183e-01,  5.84472977e-02 ... -3.40282347e+38, -3.40282347e+38, -3.40282347e+38],
 ...
 [ 9.79795009e-02, -4.86325622e-02, -3.98367018e-01 ... -1.25764713e-01, -3.40282347e+38, -3.40282347e+38],
 [ 1.87330201e-01, -8.79423544e-02, -3.81801456e-01 ... -2.05552116e-01, -5.75759172e-01, -3.40282347e+38],
 [ 2.38193020e-01, -3.01144809e-01,  2.20017701e-01 ...  4.94687676e-01, -2.92337626e-01,  9.30684656e-02]])

![](https://jalammar.github.io/images/gpt2/transformer-attention-masked-scores-softmax.png)

In [ ]:
attn_weights = softmax(attn_weights, axis=-1)
attn_weights.shape

In [11]:
softmax = nn.Softmax(axis=-1)
attn_weights = softmax(attn_weights)
attn_weights.shape

(1, 12, 10, 10)

In [12]:
attn_weights[0, 0]

Tensor(shape=[10, 10], dtype=Float32, value=
[[ 1.00000000e+00,  0.00000000e+00,  0.00000000e+00 ...  0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
 [ 5.79672694e-01,  4.20327336e-01,  0.00000000e+00 ...  0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
 [ 2.96906650e-01,  3.00038397e-01,  4.03054953e-01 ...  0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
 ...
 [ 1.21048786e-01,  1.04541227e-01,  7.36885294e-02 ...  9.67808738e-02,  0.00000000e+00,  0.00000000e+00],
 [ 1.56664565e-01,  1.18965611e-01,  8.86747688e-02 ...  1.05765536e-01,  7.30407611e-02,  0.00000000e+00],
 [ 1.20576210e-01,  7.03121126e-02,  1.18404493e-01 ...  1.55831710e-01,  7.09341019e-02,  1.04288116e-01]])

![](https://jalammar.github.io/images/gpt2/gpt2-self-attention-multihead-sum-1.png)

In [13]:
attn_output = ops.matmul(attn_weights, value)

attn_output.shape

(1, 12, 10, 64)

## GPT-2 Self-attention: 3.5- Merge attention heads

![](https://jalammar.github.io/images/gpt2/gpt2-self-attention-merge-heads-1.png)

In [14]:
def merge_heads(tensor, num_heads, attn_head_size):
    """
    Merges attn_head_size dim and num_attn_heads dim into hidden_size
    """
    # (batch_size, num_heads, seq_len, attn_head_size) --> (batch_size, seq_len, num_heads, seq_len)
    tensor = ops.transpose(tensor, (0, 2, 1, 3))
    new_shape = tensor.shape[:-2] + (num_heads * attn_head_size,)
    return tensor.view(new_shape)

In [15]:
# (1, 12, 10, 64) --> (1, 10, 12, 64) --> (1, 10, 768)
attn_output = merge_heads(attn_output, num_heads, head_dim)

attn_output.shape

(1, 10, 768)

## GPT-2 Self-attention: 4- Projecting

![](https://jalammar.github.io/images/gpt2/gpt2-self-attention-project-1.png)

In [16]:
c_proj = Dense(embed_dim, embed_dim)

In [17]:
attn_output = c_proj(attn_output)
attn_output.shape

(1, 10, 768)